# MTHexapod - Timeline

In [25]:
start_time = "2025-10-17T16:18"
end_time = "2025-10-17T18:00"
sal_index = 2
min_log_level = 40

In [2]:
import numpy as np
import pandas as pd

from astropy.time import Time
from datetime import timedelta

from bokeh.plotting import figure, show
from bokeh.models import (
    ColumnDataSource,
    CDSView,
    BooleanFilter,
    HoverTool,
    FactorRange,
    Legend,
    Span,
    CustomJS,
    RangeTool,
)
from bokeh.layouts import column, gridplot
from bokeh.io import output_notebook
from bokeh.models import FixedTicker
from bokeh.palettes import Category10_10

from lsst_efd_client import EfdClient
from lsst.summit.utils.efdUtils import (
    getEfdData,
    getDayObsEndTime,
    getDayObsStartTime,
    makeEfdClient,
)
from lsst.ts.xml.enums.MTHexapod import ApplicationStatus, EnabledSubstate, ErrorCode

In [3]:
# Initialize an EFD client
efd_client = makeEfdClient()

# Global variables
HEXAPOD_AXES = ["x", "y", "z", "u", "v", "w"]
N_AXES = len(HEXAPOD_AXES)
N_STRUTS = 6

# We want to see all the data in a column
pd.set_option("display.max_colwidth", None)

# Convert timestamps
t_start = Time(start_time, format="isot", scale="utc")
t_end = Time(end_time, format="isot", scale="utc")

# Ensures bokeh works
output_notebook()

Loading BokehJS ...

## Query

In [4]:
motor_currents = getEfdData(
    efd_client,
    topic="lsst.sal.MTHexapod.electrical",
    columns=[f"motorCurrent{i}" for i in range(N_STRUTS)] + ["salIndex"],
    begin=t_start,
    end=t_end,
)

motor_currents = motor_currents[motor_currents["salIndex"] == sal_index]
motor_currents.drop(columns=["salIndex"], inplace=True)
motor_currents["time"] = motor_currents.index

In [5]:
cmd_move = getEfdData(
    efd_client,
    topic="lsst.sal.MTHexapod.command_move",
    columns=HEXAPOD_AXES + ["salIndex", "sync"],
    begin=t_start,
    end=t_end,
)

try:
    cmd_move = cmd_move[cmd_move["salIndex"] == sal_index]
    cmd_move.drop(columns=["salIndex"], inplace=True)
except KeyError:
    cmd_move = pd.DataFrame(columns=HEXAPOD_AXES + ["sync"])

In [7]:
cmd_move_in_steps = getEfdData(
    efd_client,
    topic="lsst.sal.MTHexapod.command_moveInSteps",
    columns="*",
    begin=t_start,
    end=t_end,
)

try:
    cmd_move_in_steps = cmd_move_in_steps[cmd_move_in_steps["salIndex"] == sal_index]
    cmd_move_in_steps.drop(columns=["salIndex"], inplace=True)
except KeyError:
    cmd_move_in_steps = pd.DataFrame(columns=HEXAPOD_AXES)

tmp_df = pd.DataFrame(
    index=cmd_move_in_steps.index,
    data=["cmd_move"] * cmd_move_in_steps.index.size,
    columns=["command_name"],
)

In [8]:
evt_in_position = getEfdData(
    efd_client,
    topic="lsst.sal.MTHexapod.logevent_inPosition",
    columns=["inPosition", "salIndex"],
    begin=t_start,
    end=t_end,
)

try:
    evt_in_position = evt_in_position[evt_in_position["salIndex"] == sal_index]
    evt_in_position.drop(columns=["salIndex"], inplace=True)
except KeyError:
    evt_in_position = pd.DataFrame(columns=["inPosition"])

In [37]:
evt_log_message = getEfdData(
    efd_client,
    topic="lsst.sal.MTHexapod.logevent_logMessage",
    columns="*",
    begin=t_start,
    end=t_end,
)
print(evt_log_message.columns)

# Define minimum log level (40=ERROR, 50=CRITICAL)
min_log_level = 40

try:
    evt_log_message = evt_log_message[evt_log_message["salIndex"] == sal_index]  # Fixed: was using evt_in_position
    evt_log_message.drop(columns=["salIndex"], inplace=True)
    
    evt_log_message = evt_log_message[evt_log_message["level"] >= min_log_level]
    
except KeyError:
    evt_log_message = pd.DataFrame(columns=["name", "level", "message"])  

Index(['filePath', 'functionName', 'level', 'lineNumber', 'message', 'name',
       'private_efdStamp', 'private_identity', 'private_kafkaStamp',
       'private_origin', 'private_rcvStamp', 'private_revCode',
       'private_seqNum', 'private_sndStamp', 'process', 'salIndex',
       'timestamp', 'traceback'],
      dtype='object')


## Plot

In [38]:
# Create column data sources
source = ColumnDataSource(data=motor_currents)

# Main time series plot - THIS DEFINES THE SHARED X_RANGE
main_fig = figure(
    height=300,
    width=800,
    tools="xpan,xwheel_zoom,reset",
    x_axis_type="datetime",
    x_axis_location="above",
    window_axis="x",
    background_fill_color="#efefef",
    x_range=(t_start.to_datetime(), t_end.to_datetime()),
)

# Plot all motor currents
for i in range(N_STRUTS):
    main_fig.line(
        "time",
        f"motorCurrent{i}",
        source=source,
        legend_label=f"Motor {i}",
        color=Category10_10[i % 10],
    )

main_fig.yaxis.axis_label = "Current [A]"
main_fig.legend.location = "top_right"
main_fig.legend.click_policy = "hide"

# === EVENTS FIGURE - MUST USE main_fig.x_range ===
# Prepare log messages data
if not evt_log_message.empty:
    evt_log_message_plot = evt_log_message.reset_index()
    evt_log_message_plot.rename(columns={"index": "time"}, inplace=True)
    # Normalize level to plot position: 40->3.0, 50->3.5
    evt_log_message_plot["y_pos"] = evt_log_message_plot["level"].apply(
        lambda x: 3.0 if x == 40 else 3.5
    )
    # Add level name for display
    evt_log_message_plot["level_name"] = evt_log_message_plot["level"].apply(
        lambda x: "ERROR" if x == 40 else "CRITICAL"
    )
    log_message_source = ColumnDataSource(data=evt_log_message_plot)
else:
    log_message_source = ColumnDataSource(data={"time": [], "y_pos": [], "name": [], "level": [], "level_name": [], "message": []})

# Prepare inPosition data
if not evt_in_position.empty:
    evt_in_position_plot = evt_in_position.reset_index()
    evt_in_position_plot.rename(columns={"index": "time"}, inplace=True)
    evt_in_position_plot["value"] = evt_in_position_plot["inPosition"].astype(float)
    in_position_source = ColumnDataSource(data=evt_in_position_plot)
else:
    in_position_source = ColumnDataSource(data={"time": [], "value": []})

# Prepare commands data
if not cmd_move.empty:
    cmd_move_plot = cmd_move.reset_index()
    cmd_move_plot.rename(columns={"index": "time"}, inplace=True)
    commands_source = ColumnDataSource(data=cmd_move_plot)
else:
    commands_source = ColumnDataSource(data={"time": []})

# Prepare moveInSteps commands data
if not cmd_move_in_steps.empty:
    cmd_move_in_steps_plot = cmd_move_in_steps.reset_index()
    cmd_move_in_steps_plot.rename(columns={"index": "time"}, inplace=True)
    events_source = ColumnDataSource(data=cmd_move_in_steps_plot)
else:
    events_source = ColumnDataSource(data={"time": []})

# Create events figure with more vertical space
events_fig = figure(
    height=150,
    width=800,
    tools="xpan,xwheel_zoom,reset",
    x_axis_type="datetime",
    x_range=main_fig.x_range,
    y_range=(-0.2, 2.5),
    background_fill_color="#efefef",
)

# Plot inPosition as step line
if not evt_in_position.empty:
    events_fig.step(
        "time",
        "value",
        source=in_position_source,
        line_width=2,
        color="green",
        mode="after",
    )

# Plot cmd_move commands at y=1.5
if not cmd_move.empty:
    events_fig.segment(
        x0="time",
        y0=1.5,
        x1="time",
        y1=1.5,
        source=commands_source,
        color="blue",
        line_width=3,
    )

# Plot cmd_move_in_steps at y=2.2
if not cmd_move_in_steps.empty:
    events_circles = events_fig.scatter(
        x="time",
        y=2.2,
        size=8,
        source=events_source,
        color="red",
    )
    
    # Add hover tool to show event details
    hover = HoverTool(
        renderers=[events_circles],
        tooltips=[
            ("Time", "@time{%F %T}"),
            ("x", "@x{0.000}"),
            ("y", "@y{0.000}"),
            ("z", "@z{0.000}"),
            ("u", "@u{0.000}"),
            ("v", "@v{0.000}"),
            ("w", "@w{0.000}"),
        ],
    )
    hover.formatters = {"@time": "datetime"}
    events_fig.add_tools(hover)

# Configure y-axis with separate labels for inPosition True/False
events_fig.yaxis.ticker = [0, 1, 1.5, 2.2]
events_fig.yaxis.major_label_overrides = {
    0: "In Position: False",
    1: "In Position: True",
    1.5: "cmd_move",
    2.2: "cmd_moveInSteps",
}
events_fig.yaxis.major_label_text_font_size = "9pt"
events_fig.ygrid.grid_line_color = None


# === SELECTOR FIGURE ===
select = figure(
    title="Drag the middle and edges of the selection box to change the range above",
    height=130,
    width=800,
    x_axis_type="datetime",
    y_axis_type=None,
    tools="",
    toolbar_location=None,
    background_fill_color="#efefef",
)
select.x_range.range_padding = 0
select.x_range.bounds = "auto"

# RangeTool controls main_fig.x_range (which events_fig also shares)
range_tool = RangeTool(x_range=main_fig.x_range, start_gesture="pan")
range_tool.overlay.fill_color = "navy"
range_tool.overlay.fill_alpha = 0.2

for i in range(N_STRUTS):
    select.line(
        "time",
        f"motorCurrent{i}",
        source=source,
        color=Category10_10[i % 10],
    )

select.ygrid.grid_line_color = None
select.add_tools(range_tool)

# Display all three figures
show(column(main_fig, events_fig, select))